In [11]:
# Cell 1: Import và hàm đếm tham số
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary as model_summary   # pip install torchinfo

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("CELL 1: Sẵn sàng thiết kế mô hình")

CELL 1: Sẵn sàng thiết kế mô hình


In [12]:
# Cell 2: Định nghĩa lại các block (ResidualBlock, SEBlock, ...) — giống Notebook 2
class ResidualBlock2D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.shortcut = (nn.Identity() if (stride == 1 and in_c == out_c)
                         else nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                                            nn.BatchNorm2d(out_c)))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResidualBlock1D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_c)
        self.conv2 = nn.Conv1d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_c)
        self.shortcut = (nn.Identity() if (stride == 1 and in_c == out_c)
                         else nn.Sequential(nn.Conv1d(in_c, out_c, 1, stride, bias=False),
                                            nn.BatchNorm1d(out_c)))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class SEBlock2D(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 4)
        self.sq = nn.AdaptiveAvgPool2d(1)
        self.ex = nn.Sequential(nn.Linear(c, h, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(h, c, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.ex(self.sq(x).view(b, c)).view(b, c, 1, 1)
        return x * s

class SEBlock1D(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 4)
        self.sq = nn.AdaptiveAvgPool1d(1)
        self.ex = nn.Sequential(nn.Linear(c, h, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(h, c, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.shape
        s = self.ex(self.sq(x).view(b, c)).view(b, c, 1)
        return x * s

print("Đã định nghĩa các block.")

Đã định nghĩa các block.


In [13]:
# Cell 3: M1 - Baseline
class M1_BasicCNN2D(nn.Module):
    def __init__(self, in_ch=3, n_cls=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),    nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, n_cls))
    def forward(self, x): return self.classifier(self.features(x))

class M1_BasicCNN1D(nn.Module):
    def __init__(self, n_cls=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 3, padding=1), nn.ReLU(inplace=True))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(32, n_cls))
    def forward(self, x): return self.classifier(self.features(x))

print(f"M1 2D params: {count_params(M1_BasicCNN2D(3, 10)):,}")
print(f"M1 1D params: {count_params(M1_BasicCNN1D(2)):,}")

M1 2D params: 20,042
M1 1D params: 1,698


In [14]:
# Cell 4: M2 - BatchNorm + Dropout
class M2_CNN2D(nn.Module):
    def __init__(self, in_ch=3, n_cls=10, drop=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2), nn.Dropout2d(drop),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2), nn.Dropout2d(drop))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, 128, bias=False), nn.BatchNorm1d(128),
            nn.ReLU(inplace=True), nn.Dropout(drop), nn.Linear(128, n_cls))
    def forward(self, x): return self.classifier(self.features(x))

class M2_CNN1D(nn.Module):
    def __init__(self, n_cls=2, drop=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, 3, padding=1, bias=False),
            nn.BatchNorm1d(16), nn.ReLU(inplace=True), nn.MaxPool1d(2), nn.Dropout(drop),
            nn.Conv1d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(32, 64, bias=False), nn.BatchNorm1d(64),
            nn.ReLU(inplace=True), nn.Dropout(drop), nn.Linear(64, n_cls))
    def forward(self, x): return self.classifier(self.features(x))

print(f"M2 2D params: {count_params(M2_CNN2D(3, 10)):,}")
print(f"M2 1D params: {count_params(M2_CNN1D(2)):,}")

M2 2D params: 29,226
M2 1D params: 3,986


In [15]:
# Cell 5: M3 - Residual
class M3_ResNet2D(nn.Module):
    def __init__(self, in_ch=3, n_cls=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, 1, 1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.layer1 = ResidualBlock2D(32, 32)
        self.pool   = nn.MaxPool2d(2)
        self.layer2 = ResidualBlock2D(32, 64)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(64, n_cls))
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.pool(x)
        x = self.layer2(x)
        return self.head(x)

class M3_ResNet1D(nn.Module):
    def __init__(self, n_cls=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 16, 3, padding=1, bias=False),
            nn.BatchNorm1d(16), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        self.layer1 = ResidualBlock1D(16, 16)
        self.pool   = nn.MaxPool1d(2)
        self.layer2 = ResidualBlock1D(16, 32)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(32, n_cls))
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.pool(x)
        x = self.layer2(x)
        return self.head(x)

print(f"M3 2D params: {count_params(M3_ResNet2D(3, 10)):,}")
print(f"M3 1D params: {count_params(M3_ResNet1D(2)):,}")

M3 2D params: 77,866
M3 1D params: 7,058


In [16]:
# Cell 6: M4 - SE-Attention
class M4_SE2D(nn.Module):
    def __init__(self, in_ch=3, n_cls=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), SEBlock2D(32), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), SEBlock2D(64), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(64, n_cls))
    def forward(self, x): return self.classifier(self.features(x))

class M4_SE1D(nn.Module):
    def __init__(self, n_cls=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, 3, padding=1, bias=False),
            nn.BatchNorm1d(16), nn.ReLU(inplace=True), SEBlock1D(16), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True), SEBlock1D(32), nn.MaxPool1d(2))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(32, n_cls))
    def forward(self, x): return self.classifier(self.features(x))

print(f"M4 2D params: {count_params(M4_SE2D(3, 10)):,}")
print(f"M4 1D params: {count_params(M4_SE1D(2)):,}")

M4 2D params: 20,906
M4 1D params: 2,130


In [17]:
# Cell 7: Bảng so sánh 4 mô hình
import pandas as pd
data = []
for name, m in [('M1_Basic', M1_BasicCNN2D(3,10)),
                ('M2_BN_Drop', M2_CNN2D(3,10)),
                ('M3_Residual', M3_ResNet2D(3,10)),
                ('M4_SE', M4_SE2D(3,10))]:
    data.append([name, 'CNN 2D', count_params(m)])
for name, m in [('M1_Basic', M1_BasicCNN1D(2)),
                ('M2_BN_Drop', M2_CNN1D(2)),
                ('M3_Residual', M3_ResNet1D(2)),
                ('M4_SE', M4_SE1D(2))]:
    data.append([name, 'CNN 1D', count_params(m)])
df_arch = pd.DataFrame(data, columns=['Mô hình', 'Loại', 'Số tham số'])
print(df_arch.to_string(index=False))

    Mô hình   Loại  Số tham số
   M1_Basic CNN 2D       20042
 M2_BN_Drop CNN 2D       29226
M3_Residual CNN 2D       77866
      M4_SE CNN 2D       20906
   M1_Basic CNN 1D        1698
 M2_BN_Drop CNN 1D        3986
M3_Residual CNN 1D        7058
      M4_SE CNN 1D        2130


In [18]:
# Cell 8: Sanity check forward
x2d = torch.randn(4, 3, 32, 32)
x1d = torch.randn(4, 1, 8)

for name, m in [('M1', M1_BasicCNN2D(3,10)), ('M2', M2_CNN2D(3,10)),
                ('M3', M3_ResNet2D(3,10)), ('M4', M4_SE2D(3,10))]:
    print(f"{name} 2D: in={tuple(x2d.shape)} -> out={tuple(m(x2d).shape)}")

for name, m in [('M1', M1_BasicCNN1D(2)), ('M2', M2_CNN1D(2)),
                ('M3', M3_ResNet1D(2)), ('M4', M4_SE1D(2))]:
    print(f"{name} 1D: in={tuple(x1d.shape)} -> out={tuple(m(x1d).shape)}")

M1 2D: in=(4, 3, 32, 32) -> out=(4, 10)
M2 2D: in=(4, 3, 32, 32) -> out=(4, 10)
M3 2D: in=(4, 3, 32, 32) -> out=(4, 10)
M4 2D: in=(4, 3, 32, 32) -> out=(4, 10)
M1 1D: in=(4, 1, 8) -> out=(4, 2)
M2 1D: in=(4, 1, 8) -> out=(4, 2)
M3 1D: in=(4, 1, 8) -> out=(4, 2)
M4 1D: in=(4, 1, 8) -> out=(4, 2)
